In [ ]:
import geopandas as gpd


#create lookup between 2021 OA and 2011 LSOAs using popweighted centroids

lsoa_2011= gpd.read_file("../data/geofiles/Lower_layer_Super_Output_Areas_Dec_2011_Boundaries_Full_Clipped_BFC_EW_V3_2022_-5365225720680633795.gpkg")
oa_2021 = gpd.read_file("../data/geofiles/Output_Areas_(December_2021)_Boundaries_EW_BFE_(V9)_and_RUC.geojson")

#create lookup between 2021 OA and 2011 LSOAs bounds 
lsoa_2011 = lsoa_2011[['LSOA11CD','geometry']]
oa_2021 = oa_2021[['OA21CD','geometry']]


# Ensure both datasets use the same CRS
oa_2021 = oa_2021.to_crs(lsoa_2011.crs)
# Step 1: intersect OAs with LSOAs
overlap = gpd.overlay(oa_2021, lsoa_2011, how='intersection')

# Step 2: calculate area of each intersected polygon
overlap['overlap_area'] = overlap.geometry.area

# Step 3: for each OA, keep the LSOA with the largest overlap
overlap = overlap.sort_values('overlap_area', ascending=False)
largest_overlap = overlap.drop_duplicates(subset='OA21CD')

# Step 4: assign the LSOA code back to original OA GeoDataFrame
oa_2021 = oa_2021.drop(columns=['LSOA11CD'], errors='ignore')  # clean if already merged
oa_2021 = oa_2021.merge(largest_overlap[['OA21CD', 'LSOA11CD']], on='OA21CD', how='left')

# Check for missing ones (should be zero if geometries are good)
print("Unmatched OAs:", oa_2021['LSOA11CD'].isna().sum())
oa_2021[['OA21CD', 'LSOA11CD']].to_csv("../data/lookup_oa2022_lsoa11_EW.csv", index=False)

Unmatched OAs: 0
